In [ ]:
# IMPORTS
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Set to directories containing each model's index_record.txt and snapshots
RESULTS_DIR_FINITE = os.path.join(os.getcwd(), 'discrete')
RESULTS_DIR_CONTINUOUS = os.path.join(os.getcwd(), 'continuous')

def snapshot_path_discrete(run_index, results_dir):
    base = os.path.join(results_dir, f'snapshots{run_index}')
    return os.path.join(base, f'snapshots{run_index}_s1.h5')

def snapshot_path_continuous(run_index, results_dir):
    base = os.path.join(results_dir, f'snapshots_cts{run_index}')
    return os.path.join(base, f'snapshots_cts{run_index}_s1.h5')



In [ ]:
# READ FINITE (DISCRETE) SIMS
Ma_indices_f = []
Ma_list_f = []
Ma_f_max = 1e7 - 1

index_file = os.path.join(RESULTS_DIR_FINITE, 'index_record.txt')
with open(index_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            index_str, param_str = line.split()
            if float(param_str) > Ma_f_max:
                continue
            Ma_indices_f.append(int(index_str))
            Ma_list_f.append(float(param_str))

print('Indices (finite):', Ma_indices_f)
print('Ma list (finite):', Ma_list_f)

success_bools_f = []
final_depths_f = []
final_times_f = []

for i in range(len(Ma_list_f)):
    try:
        print('Processing Ma=' + str(Ma_list_f[i]))
        filepath = snapshot_path_discrete(Ma_indices_f[i], RESULTS_DIR_FINITE)
        with h5py.File(filepath, 'r') as f:
            s_data = f['tasks']['zb'][:]
            times = f['scales']['sim_time'][:]
        if s_data[-1] > 0:
            success_bools_f.append(True)
        else:
            success_bools_f.append(False)
        if success_bools_f[-1]:
            final_depths_f.append(0.0)
        else:
            final_depths_f.append(float(-s_data[-1, 0, 0, 0]))
        if success_bools_f[-1]:
            final_times_f.append(float(np.interp(0, s_data[:, 0, 0, 0], times)))
        else:
            final_times_f.append(np.inf)
    except OSError:
        print(str(Ma_list_f[i]) + ' data not found')
        final_times_f.append(np.nan)
        success_bools_f.append(np.nan)
        final_depths_f.append(np.nan)

# READ CONTINUOUS SIMS
Ma_indices_c = []
Ma_list_c = []
Ma_c_max = 1e7 - 2

index_file = os.path.join(RESULTS_DIR_CONTINUOUS, 'index_record.txt')
with open(index_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            index_str, param_str = line.split()
            if float(param_str) > Ma_c_max:
                continue
            Ma_indices_c.append(int(index_str))
            Ma_list_c.append(float(param_str))

print('Indices (continuous):', Ma_indices_c)
print('Ma list (continuous):', Ma_list_c)

success_bools_c = []
final_depths_c = []
final_times_c = []

for i in range(len(Ma_list_c)):
    try:
        print('Processing Ma=' + str(Ma_list_c[i]))
        filepath = snapshot_path_continuous(Ma_indices_c[i], RESULTS_DIR_CONTINUOUS)
        with h5py.File(filepath, 'r') as f:
            s_data = f['tasks']['zb'][:]
            times = f['scales']['sim_time'][:]
        if s_data[-1] > 0:
            success_bools_c.append(True)
        else:
            success_bools_c.append(False)
        if success_bools_c[-1]:
            final_depths_c.append(0.0)
        else:
            final_depths_c.append(float(-s_data[-1, 0, 0, 0]))
        if success_bools_c[-1]:
            final_times_c.append(float(np.interp(0, s_data[:, 0, 0, 0], times)))
        else:
            final_times_c.append(np.inf)
    except OSError:
        print(str(Ma_list_c[i]) + ' data not found')
        final_times_c.append(np.nan)
        success_bools_c.append(np.nan)
        final_depths_c.append(np.nan)



In [ ]:
# PROCESSING VISUALISATIONS
sorted_idx_f = np.argsort(Ma_list_f)
sorted_idx_c = np.argsort(Ma_list_c)
Ma_list_f_sorted = np.array(Ma_list_f)[sorted_idx_f]
Ma_list_c_sorted = np.array(Ma_list_c)[sorted_idx_c]
final_times_f_sorted = np.array(final_times_f)[sorted_idx_f]
final_times_c_sorted = np.array(final_times_c)[sorted_idx_c]
                        
Ma_f_comp = [f*g for f,g in zip(final_times_c_sorted, Ma_list_c_sorted)]
T_f_comp = [np.interp(Ma,Ma_list_f_sorted, final_times_f_sorted) for Ma in Ma_f_comp]

fig1 = plt.figure()
plt.scatter(Ma_list_c_sorted, T_f_comp,marker='d') 
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')
plt.xlabel("Ma_c")
plt.ylabel("T*_comp")
plt.title("Equal mass - cts Ma vs comparative T*")

fig2 = plt.figure(figsize=(6,6))
plt.scatter(final_times_c_sorted, T_f_comp,marker='.') 
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')
plt.xlabel("T*_c")
plt.ylabel("T*_f comp")
axlims = plt.gca().get_xlim()
x = np.linspace(10**-3,10**1.5)
plt.plot(x,x,linestyle='--')
#plt.plot(x,x/3,linestyle=':')
#plt.plot(x,x/2,linestyle=':')
plt.gca().set_xlim(5e-3,5e0)
plt.gca().set_ylim(5e-3,5e0)
plt.title("Equal mass - T*_c vs comparative T*_f")
plt.savefig("comparative_scatter.svg")

fig5 = plt.figure(figsize=(4,4))
plt.scatter(final_times_c_sorted, T_f_comp,marker='d' ) 
#plt.gca().set_xscale('log')
#plt.gca().set_yscale('log')
plt.xlabel("T*_c")
plt.ylabel("T*_f comp")
axlims = plt.gca().get_xlim()
x = np.linspace(10**-2,10**1.5)
plt.plot(x,x,linestyle='--')
plt.gca().set_xlim(5e-3,5e0)
plt.gca().set_ylim(5e-3,5e0)
plt.title("Equal mass - T*_c vs comparative T*_f")

fig3 = plt.figure()
plt.scatter(Ma_list_c_sorted, Ma_f_comp,marker='d')
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')
plt.xlabel("Ma_c")
plt.ylabel("Ma_f comp")
plt.title("Equal mass - Ma_c vs comparative Ma_f")

fig4 = plt.figure()
plt.scatter(Ma_f_comp, T_f_comp,marker='.' ) 
plt.scatter(Ma_f_comp, final_times_c_sorted,marker='.' ) 
plt.plot([8*np.pi,8*np.pi],[1e-6,1e6],linestyle='--',c='#1f77b4')
plt.plot([75,75],[1e-6,1e6],linestyle='--',c='#ff7f0e')
plt.legend(["Scaled finite emission","Continuous emission"])
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')
plt.ylim(1e-3,1e2)
plt.xlim(1e0,1e6)
plt.xlabel("Ma_f == effective mass")
plt.ylabel("T*")
plt.title("Equal mass - cts Ma vs comparative T*, with T*_c")
plt.savefig("comparative_scatter2.svg")

fig5 = plt.figure()
plt.scatter(Ma_list_c_sorted, T_f_comp,marker='.' ) 
plt.scatter(Ma_list_c_sorted, final_times_c_sorted,marker='.' ) 
plt.legend(["Scaled finite emission","Continuous emission"])
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')
plt.xlabel("Ma_c == effective rate")
plt.ylabel("T*")
plt.title("Equal mass - cts Ma vs comparative T*, with T*_c")
plt.savefig("comparative_scatter3.svg")


In [ ]:
plt.scatter(Ma_list_c,final_times_c)
plt.scatter(Ma_list_f,final_times_f)
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')


In [ ]:
plt.figure()
plt.scatter(Ma_list_f_sorted, final_times_f_sorted,marker='d') 
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')
plt.xlabel("Ma_c")
plt.ylabel("T*_comp")
plt.title("Equal mass - cts Ma vs comparative T*")
